# Part B — FreshCart ELT on Snowflake

Run every cell top to bottom. Each section is labelled to match the assignment
(B1–B5). Screenshot the output of each cell as you go for the report appendix.

**Note:** the mock dataset in `freshcart_lib.py` was built without the original
course notebook (not available when this was put together), so the dollar
figures here are illustrative, not the instructor's exact numbers — but the
*shape* of the pipeline (8 raw rows, 5 clean, 3 quarantined, 4 after dedup,
SCD2 tier changes, idempotent MERGE) matches what the assignment asks you to
trace. Copy the numbers *your* run produces into the report template.

Note: Snowflake returns unquoted column names in UPPERCASE (`ORDER_ID`, not
`order_id`), and ARRAY-typed columns come back from `fetch_pandas_all()` as a
JSON-formatted *string* (e.g. `'[]'`), not a native Python list — these cells
account for both.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

import freshcart_lib as fc

conn = fc.get_connection()
fc.run_ddl(conn)
print("Connected and DDL applied.")

/Users/sriprada/Library/Python/3.9/lib/python/site-packages/snowflake/connector/vendored/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sriprada/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Connected and DDL applied.


## B1 — Run the baseline

In [2]:
import json

fc.run_baseline_load(conn)
fc.run_transform(conn, include_currency_check=False)

bronze_count = fc.query_df(conn, f"SELECT COUNT(*) AS n FROM {fc.DATABASE}.BRONZE.ORDERS_RAW")
checked = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_CHECKED ORDER BY order_id, source")
# REASONS is a Snowflake ARRAY, which fetch_pandas_all() returns as a JSON *string* (e.g. '[]'),
# so we have to json.loads() it to get its real length -- len() on the string itself is wrong.
reason_lens = checked["REASONS"].apply(lambda s: len(json.loads(s)))
clean_count = (reason_lens == 0).sum() if len(checked) else 0
quarantine = checked[reason_lens > 0]
dedup = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_DEDUP ORDER BY order_id")
gold = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.GOLD.FCT_ORDERS ORDER BY order_id")

print("Raw rows in bronze:", bronze_count.iloc[0, 0])
print("Rows that pass cleaning:", clean_count)
print("Rows in quarantine:", len(quarantine))
print("Orders after deduplication:", len(dedup))
print("Gold rows:", len(gold), "| revenue:", gold["AMOUNT_INR"].sum(), "| loyalty points:", gold["LOYALTY_POINTS"].sum())
print("Share of rows rejected: {:.1%}".format(len(quarantine) / bronze_count.iloc[0, 0]))
quarantine[["ORDER_ID", "SOURCE", "REASONS"]] if len(quarantine) else quarantine

Raw rows in bronze: 16
Rows that pass cleaning: 10
Rows in quarantine: 6
Orders after deduplication: 4
Gold rows: 4 | revenue: 9000.0 | loyalty points: 139
Share of rows rejected: 37.5%


,ORDER_ID,SOURCE,REASONS
8,ORD-1003,web,"[\n ""missing_amount""\n]"
9,ORD-1003,web,"[\n ""missing_amount""\n]"
10,ORD-1004,web,"[\n ""missing_amount""\n]"
11,ORD-1004,web,"[\n ""missing_amount""\n]"
14,ORD-1006,web,"[\n ""negative_amount""\n]"
15,ORD-1006,web,"[\n ""negative_amount""\n]"


In [3]:
checked  # inspect silver.orders_checked

,ORDER_ID,CUSTOMER_ID,AMOUNT,CURRENCY,ORDER_DATE,UPDATED_AT,SOURCE,REASONS
0,ORD-1001,CUST-01,1500.0,INR,2026-07-01,2026-07-01 10:05:00,mobile,[]
1,ORD-1001,CUST-01,1500.0,INR,2026-07-01,2026-07-01 10:05:00,mobile,[]
2,ORD-1001,CUST-01,1500.0,INR,2026-07-01,2026-07-01 10:00:00,web,[]
3,ORD-1001,CUST-01,1500.0,INR,2026-07-01,2026-07-01 10:00:00,web,[]
4,ORD-1002,CUST-02,2200.0,INR,2026-07-02,2026-07-02 11:30:00,web,[]
5,ORD-1002,CUST-02,2200.0,INR,2026-07-02,2026-07-02 11:30:00,web,[]
6,ORD-1003,CUST-03,2100.0,INR,2026-07-03,2026-07-03 18:40:00,mobile,[]
7,ORD-1003,CUST-03,2100.0,INR,2026-07-03,2026-07-03 18:40:00,mobile,[]
8,ORD-1003,CUST-03,NaN,INR,2026-07-03,2026-07-03 09:15:00,web,"[\n ""missing_amount""\n]"
9,ORD-1003,CUST-03,NaN,INR,2026-07-03,2026-07-03 09:15:00,web,"[\n ""missing_amount""\n]"


In [4]:
dedup  # inspect silver.orders_dedup

,ORDER_ID,CUSTOMER_ID,AMOUNT,CURRENCY,ORDER_DATE,UPDATED_AT,SOURCE,REASONS
0,ORD-1001,CUST-01,1500.0,INR,2026-07-01,2026-07-01 10:05:00,mobile,[]
1,ORD-1002,CUST-02,2200.0,INR,2026-07-02,2026-07-02 11:30:00,web,[]
2,ORD-1003,CUST-03,2100.0,INR,2026-07-03,2026-07-03 18:40:00,mobile,[]
3,ORD-1005,CUST-01,3200.0,INR,2026-07-08,2026-07-08 16:45:00,web,[]


In [5]:
gold  # inspect gold.fct_orders

,ORDER_ID,CUSTOMER_ID,AMOUNT_INR,CURRENCY,ORDER_DATE,TIER_USED,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT
0,ORD-1001,CUST-01,1500.0,INR,2026-07-01,Silver,22,dc2cbf19b3b7c8eadd2fa59e17e0080f,2026-11-16 10:22:18.265
1,ORD-1002,CUST-02,2200.0,INR,2026-07-02,Bronze,22,14c34575e605a7d9ad38d67c6ae7d94f,2026-11-16 10:22:18.265
2,ORD-1003,CUST-03,2100.0,INR,2026-07-03,Silver,31,57d781a8c252c46ee0ca3c6df2d5430f,2026-11-16 10:22:18.265
3,ORD-1005,CUST-01,3200.0,INR,2026-07-08,Gold,64,4d0cfc89e2703add99e6c9e154f7e14f,2026-11-16 10:22:18.265


## B2 — Trace one record by hand (ORD-1003, then ORD-1005 vs ORD-1001)

The queries below pull exactly what you need to write the trace: the two raw
bronze rows, how `SQL_10_ORDERS_CLEAN` maps each source's different field
names onto the same columns, which copy the dedup `QUALIFY ROW_NUMBER()` line
keeps, and the gold-table arithmetic.

In [6]:
fc.query_df(conn, f"""
    SELECT source, raw_json FROM {fc.DATABASE}.BRONZE.ORDERS_RAW
    WHERE raw_json:order_id::STRING = 'ORD-1003' OR raw_json:cust_id::STRING IS NOT NULL AND raw_json:order_id::STRING = 'ORD-1003'
""")

,SOURCE,RAW_JSON
0,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
1,mobile,"{\n ""amt"": 2100,\n ""currency"": ""INR"",\n ""cu..."
2,web,"{\n ""amount"": null,\n ""currency"": ""INR"",\n ..."
3,mobile,"{\n ""amt"": 2100,\n ""currency"": ""INR"",\n ""cu..."


In [7]:
fc.query_df(conn, f"""
    SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_CHECKED WHERE order_id = 'ORD-1003' ORDER BY updated_at
""")
# Whichever row has the latest updated_at is the one QUALIFY ROW_NUMBER() ... = 1 keeps.

,ORDER_ID,CUSTOMER_ID,AMOUNT,CURRENCY,ORDER_DATE,UPDATED_AT,SOURCE,REASONS
0,ORD-1003,CUST-03,NaN,INR,2026-07-03,2026-07-03 09:15:00,web,"[\n ""missing_amount""\n]"
1,ORD-1003,CUST-03,NaN,INR,2026-07-03,2026-07-03 09:15:00,web,"[\n ""missing_amount""\n]"
2,ORD-1003,CUST-03,2100.0,INR,2026-07-03,2026-07-03 18:40:00,mobile,[]
3,ORD-1003,CUST-03,2100.0,INR,2026-07-03,2026-07-03 18:40:00,mobile,[]


In [8]:
fc.query_df(conn, f"""
    SELECT g.*, s.effective_from, s.effective_to
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g
    JOIN {fc.DATABASE}.DIM.CUSTOMERS_SCD2 s ON s.customer_id = g.customer_id AND s.tier = g.tier_used
    WHERE g.order_id IN ('ORD-1001', 'ORD-1005', 'ORD-1003')
    ORDER BY g.order_id
""")
# amount_inr * multiplier / 100, floored, is loyalty_points -- the SCD2 join (dim.customers_scd2,
# matched on order_date BETWEEN effective_from AND effective_to) is why ORD-1001 prices at
# Priya's Silver tier and ORD-1005 at her later Gold tier.

,ORDER_ID,CUSTOMER_ID,AMOUNT_INR,CURRENCY,ORDER_DATE,TIER_USED,LOYALTY_POINTS,ROW_HASH,PUBLISHED_AT,EFFECTIVE_FROM,EFFECTIVE_TO
0,ORD-1001,CUST-01,1500.0,INR,2026-07-01,Silver,22,dc2cbf19b3b7c8eadd2fa59e17e0080f,2026-11-16 10:22:18.265,2026-01-01,2026-07-05
1,ORD-1003,CUST-03,2100.0,INR,2026-07-03,Silver,31,57d781a8c252c46ee0ca3c6df2d5430f,2026-11-16 10:22:18.265,2026-01-01,9999-12-31
2,ORD-1005,CUST-01,3200.0,INR,2026-07-08,Gold,64,4d0cfc89e2703add99e6c9e154f7e14f,2026-11-16 10:22:18.265,2026-07-05,9999-12-31


## B3 — Add a validation rule (unsupported_currency)

`include_currency_check=False` above is the "before" state. Re-running with
`include_currency_check=True` adds the line inside `ARRAY_CONSTRUCT_COMPACT(...)`
in `freshcart_lib.sql_10_orders_clean()` — open that function and you'll see
exactly the one line to add if you were doing this by hand in the SQL editor.

In [9]:
before = fc.get_reasons_for_order(conn, "ORD-1006")
print("Reasons for ORD-1006 BEFORE:", before)

fc.run_transform(conn, include_currency_check=True)

after = fc.get_reasons_for_order(conn, "ORD-1006")
print("Reasons for ORD-1006 AFTER: ", after)

gold_after = fc.query_df(conn, f"SELECT SUM(amount_inr) AS revenue, COUNT(*) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS")
print("Gold total after adding the check:", gold_after)
# ORD-1006 was already in quarantine (negative_amount) before this change, so it does NOT add
# a second quarantine row -- ARRAY_CONSTRUCT_COMPACT just appends a second reason to the SAME
# row's `reasons` array. The gold total is unchanged because ORD-1006 was excluded from
# silver.orders_dedup (and therefore from gold) before AND after -- WHERE ARRAY_SIZE(reasons) = 0
# in SQL_20_ORDERS_DEDUP already filtered it out.

Reasons for ORD-1006 BEFORE: [('web', '[\n  "negative_amount"\n]'), ('web', '[\n  "negative_amount"\n]')]


Reasons for ORD-1006 AFTER:  [('web', '[\n  "negative_amount",\n  "unsupported_currency"\n]'), ('web', '[\n  "negative_amount",\n  "unsupported_currency"\n]')]


Gold total after adding the check:    REVENUE  N
0   9000.0  4


## B4 — Change history and idempotency (SCD2)

In [10]:
before_b4 = fc.query_df(conn, f"""
    SELECT g.order_id, g.tier_used, g.loyalty_points, g.published_at
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g WHERE g.order_id = 'ORD-1002'
""")
total_points_before = fc.query_df(conn, f"SELECT SUM(loyalty_points) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS").iloc[0, 0]
print("Before B4 -- ORD-1002:", before_b4.to_dict("records"), "| total loyalty points:", total_points_before)

# PREDICTION: Arjun's new Gold tier is effective_from 2026-08-15, which is AFTER
# ORD-1002's order_date (2026-07-02) -- so the point-in-time join should NOT pick up
# Gold for ORD-1002. Write your own prediction in the report BEFORE running this cell.

fc.add_tier_event_and_rerun(conn, "CUST-02", "Gold", "2026-08-15")

scd2_rows = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.DIM.CUSTOMERS_SCD2 WHERE customer_id = 'CUST-02'")
after_b4 = fc.query_df(conn, f"""
    SELECT g.order_id, g.tier_used, g.loyalty_points, g.published_at
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g WHERE g.order_id = 'ORD-1002'
""")
total_points_after = fc.query_df(conn, f"SELECT SUM(loyalty_points) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS").iloc[0, 0]

print("CUST-02 SCD2 rows:\n", scd2_rows)
print("After B4 -- ORD-1002:", after_b4.to_dict("records"), "| total loyalty points:", total_points_after)

Before B4 -- ORD-1002: [{'ORDER_ID': 'ORD-1002', 'TIER_USED': 'Bronze', 'LOYALTY_POINTS': 22, 'PUBLISHED_AT': Timestamp('2026-11-16 10:22:23.967000')}] | total loyalty points: 139


CUST-02 SCD2 rows:
   CUSTOMER_ID    TIER EFFECTIVE_FROM EFFECTIVE_TO  IS_CURRENT
0     CUST-02  Bronze     2026-01-01   2026-08-15       False
1     CUST-02    Gold     2026-08-15   9999-12-31        True
After B4 -- ORD-1002: [{'ORDER_ID': 'ORD-1002', 'TIER_USED': 'Bronze', 'LOYALTY_POINTS': 22, 'PUBLISHED_AT': Timestamp('2026-11-16 10:22:26.344000')}] | total loyalty points: 139


In [11]:
bronze_count_after = fc.query_df(conn, f"SELECT COUNT(*) AS n FROM {fc.DATABASE}.BRONZE.ORDERS_RAW")
print("Bronze row count after this re-run:", bronze_count_after.iloc[0, 0])
# Bronze grows by 8 every run because load_file() uses COPY INTO ... FORCE=TRUE on the SAME
# local files -- bronze is an append-only raw landing layer by design, with no dedup logic at
# all. row_hash (order_id + customer_id + amount_inr + tier) + the gold MERGE's
# "ON t.row_hash = src.row_hash" is where idempotency actually lives: if an order's business
# columns and tier haven't changed between runs, its row_hash is identical, so MERGE just
# updates published_at (or does nothing) instead of inserting a duplicate gold row. Only rows
# whose row_hash actually changed (like ORD-1002 if its tier changed) get a new published_at.

Bronze row count after this re-run: 16


## B5 — Add a data-quality check

In [12]:
checks = fc.quality_checks(conn)
print("quality_checks() --", checks)
assert all(v == 0 for v in checks.values()), "A check that should be 0 is failing!"

quality_checks() -- {'duplicate_orders_in_gold': 0, 'negative_loyalty_points': 0, 'orders_missing_tier': 0, 'clean_rows_with_null_amount': 0, 'quarantined_rows_with_no_real_reason': 0, 'future_dated_orders': 0}


In [13]:
# Make the NEW check (future_dated_orders) fail on purpose: insert a mock order dated
# in the future directly into silver.orders_dedup, show PROBLEM, then remove it again.
conn.cursor().execute(f"""
    INSERT INTO {fc.DATABASE}.SILVER.ORDERS_DEDUP
        (order_id, customer_id, amount, currency, order_date, updated_at, source, reasons)
    SELECT 'ORD-9999', 'CUST-01', 999.00, 'INR', DATEADD(day, 30, CURRENT_DATE()), CURRENT_TIMESTAMP(), 'web', ARRAY_CONSTRUCT()
""")

checks_with_problem = fc.quality_checks(conn)
print("PROBLEM ->", checks_with_problem)
assert checks_with_problem["future_dated_orders"] > 0

conn.cursor().execute(f"DELETE FROM {fc.DATABASE}.SILVER.ORDERS_DEDUP WHERE order_id = 'ORD-9999'")
checks_clean_again = fc.quality_checks(conn)
print("back to clean ->", checks_clean_again)

PROBLEM -> {'duplicate_orders_in_gold': 0, 'negative_loyalty_points': 0, 'orders_missing_tier': 0, 'clean_rows_with_null_amount': 0, 'quarantined_rows_with_no_real_reason': 0, 'future_dated_orders': 1}


back to clean -> {'duplicate_orders_in_gold': 0, 'negative_loyalty_points': 0, 'orders_missing_tier': 0, 'clean_rows_with_null_amount': 0, 'quarantined_rows_with_no_real_reason': 0, 'future_dated_orders': 0}


**Why ELT, not ETL, for FreshCart?** Snowflake's compute is cheap and
elastic, so it's more efficient to load the raw JSON as-is (schema-on-read)
and let SQL do every transform step — bronze/silver/quarantine/gold — inside
the warehouse, rather than paying for a separate transform server before
loading. It also means the original raw payloads are always available if a
transform rule turns out to be wrong (just re-run the SQL, you don't need to
re-extract).

**Where would ETL still win?** When the source system can't be trusted to
hold raw data securely or cheaply — e.g. a legal/compliance requirement to
strip or mask PII *before* it ever lands in a shared warehouse (so no raw,
unmasked customer data is ever stored even transiently), ETL's
transform-before-load model is the safer choice.